# Hope: GatedTCN Model V4 - Advanced Training

Este notebook entrena el modelo GatedTCN V4 (Gated Temporal Convolutional Network) con optimizaciones de vanguardia:
- **DWT Integration**: Haar Discrete Wavelet Transform para separación de tendencia y ruido.
- **Contrastive Pre-training**: Fase inicial auto-supervisada para representaciones robustas.
- **Multi-task Learning**: Head auxiliar de volatilidad para regularización.
- **Focal Loss + Label Smoothing**: Manejo de desbalance y sobreajuste.
- **Gradient Clipping & AdamW**: Estabilidad en el entrenamiento.
- **Métricas Expandidas**: ROC-AUC, Accuracy, F1.

In [ ]:
# Task 11: Environment Detection and Path Setup
import os
import sys
sys.path.append(os.path.abspath('../scripts'))

if 'COLAB_GPU' in os.environ:
    env = 'colab'
    data_path = "/content/ticks.csv"
    model_path = "model.onnx"
elif 'KAGGLE_URL_BASE' in os.environ:
    env = 'kaggle'
    data_path = "/kaggle/input/ticks-csv/ticks.csv"
    model_path = "/kaggle/working/model.onnx"
else:
    env = 'local'
    data_path = "data/ticks.csv"
    model_path = "model.onnx"
print(f"Running on: {env}")
print(f"Data path: {data_path}")

In [ ]:
!pip install torch==2.11.0 onnx==1.21.0 numpy==2.4.4 pandas==3.0.2 scikit-learn==1.8.0 tqdm==4.67.3 matplotlib==3.7.0 seaborn==0.12.2 onnxruntime==1.20.1


## 1. Model Architecture

In [ ]:
import torch
import torch.nn as nn
import torch.onnx
import numpy as np
import pandas as pd
import math
import os
import copy
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
from torch.utils.data import DataLoader, TensorDataset
from hope_ml.common import GatedTCNBlock, SEModule, GatedTCNV4, prepare_features, contrastive_loss, focal_loss, block_mask, CausalPadding1d


## 2. Data Preparation

In [ ]:
def load_data_from_csv(csv_path, limit=None):
    # Task 10: Environment-specific Path Handling
    search_paths = [csv_path, "data/ticks.csv", "/content/ticks.csv", "/kaggle/input/ticks-csv/ticks.csv"]
    actual_path = None
    for p in search_paths:
        if os.path.exists(p):
            actual_path = p
            break
    
    if actual_path is None:
        print("CSV not found in common paths.")
        return None
    
    print(f"Loading data from: {actual_path}")
    df = pd.read_csv(actual_path, header=None, names=['epoch', 'quote'], nrows=limit if limit is not None else None)
    return df['quote'].values.astype(np.float32)



## 3. Advanced Training

In [ ]:
csv_path = "ticks.csv"
seq_len = 32
input_dim = 8

prices = load_data_from_csv(csv_path)
if prices is not None:
    x_all, y_dir_all, y_vol_all = prepare_features(prices, seq_len=seq_len)
else:
    print("Using synthetic data.")
    x_all = torch.randn(2000, seq_len, input_dim)
    y_dir_all = torch.randint(0, 2, (2000, 1)).float()
    y_vol_all = torch.rand(2000, 1)

# The 80/20 train/validation split is temporal and intentional.
# The validation set always represents the most recent 20 percent of ticks.
# Shuffling is deliberately omitted before the split to prevent data leakage.
split = int(len(x_all) * 0.8)
train_ds = TensorDataset(x_all[:split], y_dir_all[:split], y_vol_all[:split])
val_ds = TensorDataset(x_all[split:], y_dir_all[split:], y_vol_all[split:])
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=128, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GatedTCNV4(input_dim=input_dim).to(device)
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total trainable parameters: {total_params:,}")
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)

# Phase 1: Contrastive Pre-training
print(f"Starting Phase 1: Contrastive Pre-training... (Initial LR: {optimizer.param_groups[0]['lr']:.6f})")
contrastive_epochs = 5
for epoch in range(contrastive_epochs):
    model.train()
    total_cl_loss = 0
    for bx, _, _ in train_loader:
        bx = bx.to(device)
        bx_aug1 = block_mask(bx)
        bx_aug2 = block_mask(bx)
        optimizer.zero_grad()
        f1 = model(bx_aug1, return_feat=True)
        f2 = model(bx_aug2, return_feat=True)
        loss = contrastive_loss(f1, f2)
        loss.backward()
        optimizer.step()
        total_cl_loss += loss.item()
    print(f"Pre-train Epoch {epoch}, CL Loss: {total_cl_loss/len(train_loader):.4f}")

# Phase 2: Supervised Fine-tuning
focal_gamma = 2.0
focal_smoothing = 0.05
print(f"Starting Phase 2: Supervised Fine-tuning (Gamma: {focal_gamma}, Smoothing: {focal_smoothing})...")
num_pos = torch.sum(y_dir_all[:split]).item()
if num_pos == 0:
    print("WARNING: No positive labels found in training split. Setting pos_weight to 1.0.")
    pos_weight = 1.0
else:
    pos_weight = (split - num_pos) / num_pos
from torch.optim.lr_scheduler import LambdaLR
base_lr = 0.001
warmup_epochs = 5
def lr_lambda(epoch):
    if epoch < warmup_epochs:
        return (epoch + 1) / warmup_epochs
    return 1.0
warmup_scheduler = LambdaLR(optimizer, lr_lambda=lr_lambda)
plateau_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

early_stop_patience = 5
patience_counter = 0
best_auc = 0.0
best_model = copy.deepcopy(model.state_dict())

for epoch in range(20):
    total_grad_norm = 0
    num_batches = 0
    model.train()
    for bx, by_dir, by_vol in train_loader:
        bx, by_dir, by_vol = bx.to(device), by_dir.to(device), by_vol.to(device)
        optimizer.zero_grad()
        out_dir, out_vol = model(bx)
        l_dir = focal_loss(out_dir, by_dir, torch.tensor(pos_weight).to(device), gamma=focal_gamma, smoothing=focal_smoothing)
        l_vol = nn.functional.mse_loss(out_vol, by_vol)
        loss = l_dir + 0.2 * l_vol
        loss.backward()
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        total_grad_norm += grad_norm.item()
        num_batches += 1
        optimizer.step()

    model.eval()
    vp, vt = [], []
    with torch.no_grad():
        for bx, by_dir, _ in val_loader:
            out_dir, _ = model(bx.to(device))
            vp.extend(out_dir.cpu().numpy().flatten())
            vt.extend(by_dir.cpu().numpy().flatten())
    
    vpb = np.array(vp) > 0.5
    acc = accuracy_score(vt, vpb)
    f1 = f1_score(vt, vpb)
    prec = precision_score(vt, vpb, zero_division=0)
    rec = recall_score(vt, vpb, zero_division=0)
    auc = roc_auc_score(vt, vp)
    if epoch < warmup_epochs:
        warmup_scheduler.step()
    else:
        plateau_scheduler.step(auc)
    avg_grad_norm = total_grad_norm / num_batches if num_batches > 0 else 0
    print(f"Epoch {epoch}, AUC: {auc:.4f}, Acc: {acc:.4f}, F1: {f1:.4f}, Prec: {prec:.4f}, Rec: {rec:.4f}, LR: {optimizer.param_groups[0]['lr']:.6f}, GradNorm: {avg_grad_norm:.4f}")
    if auc > best_auc:
        best_auc = auc
        best_model = copy.deepcopy(model.state_dict())
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= early_stop_patience:
            print(f"Early stopping triggered at epoch {epoch}")
            break

if 'best_model' in locals(): model.load_state_dict(best_model)
print(f"Final Best AUC: {best_auc:.4f}")

# Final evaluation for visualization
model.eval()
vp, vt = [], []
with torch.no_grad():
    for bx, by_dir, _ in val_loader:
        out_dir, _ = model(bx.to(device))
        vp.extend(out_dir.cpu().numpy().flatten())
        vt.extend(by_dir.cpu().numpy().flatten())
vpb = np.array(vp) > 0.5

## 4. Evaluation & Visualization

In [ ]:
# Task 13: Plot Training History
import matplotlib.pyplot as plt
import seaborn as sns

def plot_curves(history):
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(history['train_loss'], label='Train Loss')
    plt.plot(history['val_auc'], label='Val AUC')
    plt.title("Loss and AUC Progression")
    plt.legend()
    
    plt.subplot(1, 2, 2)
    plt.plot(history['cl_loss'], label='Contrastive Loss')
    plt.title("Phase 1: Contrastive Loss")
    plt.legend()
    plt.show()

print("Visualizing training progression...")


In [ ]:
# Task 12: Confusion Matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
try:
    cm = confusion_matrix(vt, vpb)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot(cmap=plt.cm.Blues)
    plt.title("Confusion Matrix")
    plt.show()
except NameError:
    print("Validation variables not found. Run training first.")

## 5. Export

In [ ]:
model.eval()

# Export wrapper to exclude auxiliary head
class InferenceModel(nn.Module):
    def __init__(self, trained_model):
        super().__init__()
        self.model = trained_model
    def forward(self, x):
        direction, _ = self.model(x)
        return direction

infer_model = InferenceModel(model)
infer_model.eval()
dummy = torch.randn(1, 32, 8).to(device)
torch.onnx.export(
    infer_model, dummy, "model.onnx", 
    export_params=True, opset_version=11, do_constant_folding=True, 
    input_names=['input'], output_names=['output']
    # Note: Model is exported with a static batch size of 1. No dynamic axes are required.
)


# Task 14: ONNX Output Validation
import onnxruntime as ort
sess = ort.InferenceSession("model.onnx")
out = sess.run(None, {'input': dummy.cpu().numpy()})[0]
assert out.shape == (1, 1), f"Unexpected output shape: {out.shape}"
assert 0.0 <= out.item() <= 1.0, f"Output out of bounds: {out.item()}"
print(f"ONNX Model validation successful. Output: {out.item():.4f}")

# Task 13: Environment-aware Export
if env == 'colab':
    from google.colab import files
    files.download("model.onnx")
elif env == 'kaggle':
    print("Model saved to /kaggle/working/model.onnx")
else:
    print("Model saved locally as model.onnx")

# Task 9: INT8 Quantization
quantization_succeeded = False
try:
    from onnxruntime.quantization import quantize_dynamic, QuantType
    quant_path = "model_quantized.onnx"
    quantize_dynamic("model.onnx", quant_path, weight_type=QuantType.QUInt8)
    print(f"Dynamic INT8 Quantization Successful: {quant_path}")
    quantization_succeeded = True
except Exception as e:
    print(f"Quantization skipped or failed: {e}")
